# Iniciar LLM de Torres Quevedo con Ollama en Google Colab

**Importante antes de ejecutar:**
1. Cambiar tipo de entorno de ejecución → `GPU T4`
2. Ejecuta las celdas **en orden**


## 1. Instalar Ollama

In [ ]:
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh
print('✅ Ollama instalado')

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (416 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently i

## 2. Iniciar Ollama

In [ ]:
import subprocess, time, os, requests

env = {**os.environ, 'OLLAMA_HOST': '0.0.0.0:11434'}
subprocess.Popen(['ollama', 'serve'], env=env,
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print('⏳ Iniciando Ollama...')
for i in range(15):
    time.sleep(1)
    try:
        if requests.get('http://localhost:11434').status_code == 200:
            print('✅ Ollama corriendo en localhost:11434')
            break
    except:
        print(f'  Esperando... ({i+1}s)')
else:
    print('❌ Ollama no arrancó, ejecuta la celda de nuevo')

⏳ Iniciando Ollama...
✅ Ollama corriendo en localhost:11434


## 3. importar Modelfile

In [ ]:
from google.colab import files

MODELFILE_NAME = None

print('📂 Selecciona tu Modelfile...')

uploaded = files.upload()
for filename in uploaded.keys():
    print(f'✅ Archivo subido: {filename}')
    MODELFILE_NAME = filename

📂 Selecciona tu Modelfile...


Saving modelfile.mf to modelfile (1).mf
✅ Archivo subido: modelfile (1).mf


## 4. Crear modelo desde tu Modelfile

Por defecto el modelo que se crea se llama `torresQuevedoLLM`

In [ ]:
import subprocess

MODEL_NAME = 'torresQuevedoLLM'

print(f'⏳ Creando modelo {MODEL_NAME} con contexto {MODELFILE_NAME}...')

result = subprocess.run(
    ['ollama', 'create', MODEL_NAME, '-f', MODELFILE_NAME],
    capture_output=True, text=True
)

print(result.stdout)

if result.returncode == 0:
    print(f'✅ Modelo {MODEL_NAME} listo')
else:
    print(f'❌ Error: {result.stderr}')

⏳ Creando modelo torresQuevedoLLM con contexto modelfile (1).mf...

✅ Modelo torresQuevedoLLM listo


## 5. Verificar que el modelo responde

In [ ]:
import requests

payload = {
    'model': MODEL_NAME,
    'messages': [{'role': 'user', 'content': 'Hola, responde en una frase corta'}],
    'stream': False
}

print('⏳ Enviando prompt de prueba...')
r = requests.post('http://localhost:11434/api/chat', json=payload)

if r.status_code == 200:
    print(f'✅ Respuesta: {r.json()["message"]["content"]}')
else:
    print(f'❌ Error {r.status_code}: {r.text}')

⏳ Enviando prompt de prueba...
✅ Respuesta: ¡Hola! Mucho gusto, soy un agricultor con experiencia trabajando al aire libre.


## 6. Exponer con Cloudflare Tunnel

In [ ]:
import subprocess, threading, time, re

# Instalar cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print('✅ cloudflared instalado')

# Variable para guardar la URL
tunnel_url = {'value': None}

def run_tunnel():
    process = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        # Buscar la URL en los logs de cloudflared
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match and tunnel_url['value'] is None:
            tunnel_url['value'] = match.group(0)

# Lanzar el túnel en un hilo separado
t = threading.Thread(target=run_tunnel, daemon=True)
t.start()

# Esperar a que aparezca la URL
print('⏳ Generando túnel Cloudflare...')
for i in range(30):
    time.sleep(1)
    if tunnel_url['value']:
        unity_url = tunnel_url['value'] + '/api/chat'
        print('\n' + '='*60)
        print('✅ TÚNEL ACTIVO')
        print('='*60)
        print(f'\n📋 URL endpoint:')
        print(f'\n   {unity_url}\n')
        print('='*60)
        print('⚠️  Colab no debe cerrarse para mantener la conexión abierta.')
        break
else:
    print('❌ No se pudo obtener la URL. Ejecuta la celda de nuevo.')

✅ cloudflared instalado
⏳ Generando túnel Cloudflare...

✅ TÚNEL ACTIVO

📋 URL endpoint:

   https://copyrighted-produces-risks-non.trycloudflare.com/api/chat

⚠️  Colab no debe cerrarse para mantener la conexión abierta.


## 7. Mantener la sesión

In [ ]:
import time, requests

print('🔄 Menteniendo conexion.')
print(f'   URL activa: {tunnel_url["value"]}/api/chat\n')

ping_count = 0
while True:
    try:
        r = requests.get('http://localhost:11434')
        ping_count += 1
        print(f'  ✅ Ping #{ping_count} — Ollama OK — {time.strftime("%H:%M:%S")}')
    except Exception as e:
        print(f'  ❌ Ollama no responde: {e}')
    time.sleep(30)

🔄 Menteniendo conexion.
   URL activa: https://copyrighted-produces-risks-non.trycloudflare.com/api/chat

  ✅ Ping #1 — Ollama OK — 08:03:03
  ✅ Ping #2 — Ollama OK — 08:03:33
  ✅ Ping #3 — Ollama OK — 08:04:03
  ✅ Ping #4 — Ollama OK — 08:04:33
  ✅ Ping #5 — Ollama OK — 08:05:03
  ✅ Ping #6 — Ollama OK — 08:05:33
  ✅ Ping #7 — Ollama OK — 08:06:03
  ✅ Ping #8 — Ollama OK — 08:06:33
  ✅ Ping #9 — Ollama OK — 08:07:03
  ✅ Ping #10 — Ollama OK — 08:07:33
  ✅ Ping #11 — Ollama OK — 08:08:03
  ✅ Ping #12 — Ollama OK — 08:08:33
  ✅ Ping #13 — Ollama OK — 08:09:03
  ✅ Ping #14 — Ollama OK — 08:09:33
  ✅ Ping #15 — Ollama OK — 08:10:03
  ✅ Ping #16 — Ollama OK — 08:10:33
  ✅ Ping #17 — Ollama OK — 08:11:03
  ✅ Ping #18 — Ollama OK — 08:11:33
  ✅ Ping #19 — Ollama OK — 08:12:03
  ✅ Ping #20 — Ollama OK — 08:12:33
  ✅ Ping #21 — Ollama OK — 08:13:03
  ✅ Ping #22 — Ollama OK — 08:13:33
  ✅ Ping #23 — Ollama OK — 08:14:03
  ✅ Ping #24 — Ollama OK — 08:14:33


KeyboardInterrupt: 